###Processed E-commerce Dataset

This notebook combines the **Orders**, **Customers**, and **Products** CSV files using Pandas.




In [ ]:
import pandas as pd
from pathlib import Path

# Keep the CSV files in the same folder as this notebook.
DATA_DIR = Path(".")

orders = pd.read_csv(DATA_DIR / "Day9_Orders.csv")
customers = pd.read_csv(DATA_DIR / "Day9_Customers.csv")
products = pd.read_csv(DATA_DIR / "Day9_Products.csv")

print("Orders:", orders.shape)
print("Customers:", customers.shape)
print("Products:", products.shape)


Orders: (120, 7)
Customers: (30, 5)
Products: (20, 5)


In [2]:
print("Orders")
display(orders.head())

print("Customers")
display(customers.head())

print("Products")
display(products.head())


Orders


,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Payment_Method,Order_Status
0,O0001,2026-02-19,C027,P019,2,Credit Card,Delivered
1,O0002,2026-01-25,C006,P003,2,Debit Card,Delivered
2,O0003,2026-02-26,C015,P004,1,Cash on Delivery,Delivered
3,O0004,2026-03-04,C024,P015,3,Net Banking,Delivered
4,O0005,2026-03-29,C025,P009,5,Credit Card,Delivered


Customers


,Customer_ID,Customer_Name,City,Region,Membership_Type
0,C001,Aarav Sharma,Srinagar,North,Premium
1,C002,Zoya Khan,Delhi,North,Regular
2,C003,Rohan Mehta,Mumbai,West,Premium
3,C004,Ananya Singh,Jammu,North,Regular
4,C005,Kabir Ali,Lucknow,North,New


Products


,Product_ID,Product_Name,Category,Unit_Price,Brand
0,P001,Wireless Headphones,Electronics,1499,SoundMax
1,P002,Mechanical Keyboard,Electronics,2499,KeyPro
2,P003,Wireless Mouse,Electronics,899,TechGear
3,P004,Smart Watch,Electronics,3299,FitTech
4,P005,Power Bank,Electronics,1199,VoltPlus


## 1. Data inspection

Before combining the datasets, we check their columns and data types to identify the common keys used for merging.

In [3]:
print("Orders columns:", orders.columns.tolist())
print("Customers columns:", customers.columns.tolist())
print("Products columns:", products.columns.tolist())

print("\nOrders data types:")
print(orders.dtypes)


Orders columns: ['Order_ID', 'Order_Date', 'Customer_ID', 'Product_ID', 'Quantity', 'Payment_Method', 'Order_Status']
Customers columns: ['Customer_ID', 'Customer_Name', 'City', 'Region', 'Membership_Type']
Products columns: ['Product_ID', 'Product_Name', 'Category', 'Unit_Price', 'Brand']

Orders data types:
Order_ID          object
Order_Date        object
Customer_ID       object
Product_ID        object
Quantity           int64
Payment_Method    object
Order_Status      object
dtype: object


## 2. DateTime conversion

The `Order_Date` column is converted to Pandas datetime format so that date components can be extracted easily.

In [4]:
orders["Order_Date"] = pd.to_datetime(orders["Order_Date"], errors="coerce")

print(orders[["Order_ID", "Order_Date"]].head())
print("\nDate type:", orders["Order_Date"].dtype)


  Order_ID Order_Date
0    O0001 2026-02-19
1    O0002 2026-01-25
2    O0003 2026-02-26
3    O0004 2026-03-04
4    O0005 2026-03-29

Date type: datetime64[ns]


## 3. Demonstrating `concat()`

The Orders DataFrame is divided into two parts and then combined again using `pd.concat()`. This demonstrates vertical DataFrame concatenation.

In [5]:
mid = len(orders) // 2

orders_part1 = orders.iloc[:mid].copy()
orders_part2 = orders.iloc[mid:].copy()

orders_concat = pd.concat(
    [orders_part1, orders_part2],
    ignore_index=True
)

print("Part 1 shape:", orders_part1.shape)
print("Part 2 shape:", orders_part2.shape)
print("After concat:", orders_concat.shape)
display(orders_concat.head())


Part 1 shape: (60, 7)
Part 2 shape: (60, 7)
After concat: (120, 7)


,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Payment_Method,Order_Status
0,O0001,2026-02-19,C027,P019,2,Credit Card,Delivered
1,O0002,2026-01-25,C006,P003,2,Debit Card,Delivered
2,O0003,2026-02-26,C015,P004,1,Cash on Delivery,Delivered
3,O0004,2026-03-04,C024,P015,3,Net Banking,Delivered
4,O0005,2026-03-29,C025,P009,5,Credit Card,Delivered


## 4. Combining related information with `merge()`

Orders are first merged with Products using `Product_ID`, and then with Customers using `Customer_ID`.

In [6]:
processed = orders_concat.merge(
    products,
    on="Product_ID",
    how="left",
    validate="many_to_one"
)

processed = processed.merge(
    customers,
    on="Customer_ID",
    how="left",
    validate="many_to_one"
)

print("Combined shape:", processed.shape)
display(processed.head())


Combined shape: (120, 15)


,Order_ID,Order_Date,Customer_ID,Product_ID,Quantity,Payment_Method,Order_Status,Product_Name,Category,Unit_Price,Brand,Customer_Name,City,Region,Membership_Type
0,O0001,2026-02-19,C027,P019,2,Credit Card,Delivered,Cricket Bat,Sports,2499,BatPro,Harsh Vardhan,Noida,North,Premium
1,O0002,2026-01-25,C006,P003,2,Debit Card,Delivered,Wireless Mouse,Electronics,899,TechGear,Ishita Gupta,Bengaluru,South,Premium
2,O0003,2026-02-26,C015,P004,1,Cash on Delivery,Delivered,Smart Watch,Electronics,3299,FitTech,Karan Joshi,Chandigarh,North,Regular
3,O0004,2026-03-04,C024,P015,3,Net Banking,Delivered,Machine Learning Basics,Books,999,AIPress,Maryam Khan,Hyderabad,South,Regular
4,O0005,2026-03-29,C025,P009,5,Credit Card,Delivered,Coffee Maker,Home & Kitchen,3499,HomeBrew,Reyansh Jain,Kolkata,East,New


## 5. Creating useful columns with `apply()`

`apply()` is used to calculate total sales, classify sales levels, and create a customer label.

In [7]:
# Total value of each order line
processed["Total_Sales"] = processed.apply(
    lambda row: row["Quantity"] * row["Unit_Price"],
    axis=1
)

# Simple sales classification
processed["Sales_Level"] = processed["Total_Sales"].apply(
    lambda x: "High" if x >= 3000 else ("Medium" if x >= 1500 else "Low")
)

# Combine customer name and membership type
processed["Customer_Label"] = processed.apply(
    lambda row: f"{row['Customer_Name']} ({row['Membership_Type']})",
    axis=1
)

display(
    processed[
        ["Order_ID", "Product_Name", "Quantity", "Unit_Price",
         "Total_Sales", "Sales_Level", "Customer_Label"]
    ].head(10)
)


,Order_ID,Product_Name,Quantity,Unit_Price,Total_Sales,Sales_Level,Customer_Label
0,O0001,Cricket Bat,2,2499,4998,High,Harsh Vardhan (Premium)
1,O0002,Wireless Mouse,2,899,1798,Medium,Ishita Gupta (Premium)
2,O0003,Smart Watch,1,3299,3299,High,Karan Joshi (Regular)
3,O0004,Machine Learning Basics,3,999,2997,Medium,Maryam Khan (Regular)
4,O0005,Coffee Maker,5,3499,17495,High,Reyansh Jain (New)
5,O0006,Football,3,799,2397,Medium,Rohan Mehta (Premium)
6,O0007,Wireless Mouse,3,899,2697,Medium,Vivaan Kapoor (New)
7,O0008,Power Bank,4,1199,4796,High,Priya Menon (Premium)
8,O0009,Coffee Maker,2,3499,6998,High,Maryam Khan (Regular)
9,O0010,Power Bank,2,1199,2398,Medium,Fatima Noor (Regular)


## 6. DateTime feature extraction

The converted date is used to create year, month number, month name, day, and day-of-week columns.

In [8]:
processed["Order_Year"] = processed["Order_Date"].dt.year
processed["Order_Month"] = processed["Order_Date"].dt.month
processed["Order_Month_Name"] = processed["Order_Date"].dt.month_name()
processed["Order_Day"] = processed["Order_Date"].dt.day
processed["Day_of_Week"] = processed["Order_Date"].dt.day_name()

display(
    processed[
        ["Order_ID", "Order_Date", "Order_Year", "Order_Month",
         "Order_Month_Name", "Order_Day", "Day_of_Week"]
    ].head(10)
)


,Order_ID,Order_Date,Order_Year,Order_Month,Order_Month_Name,Order_Day,Day_of_Week
0,O0001,2026-02-19,2026,2,February,19,Thursday
1,O0002,2026-01-25,2026,1,January,25,Sunday
2,O0003,2026-02-26,2026,2,February,26,Thursday
3,O0004,2026-03-04,2026,3,March,4,Wednesday
4,O0005,2026-03-29,2026,3,March,29,Sunday
5,O0006,2026-02-09,2026,2,February,9,Monday
6,O0007,2026-02-10,2026,2,February,10,Tuesday
7,O0008,2026-03-27,2026,3,March,27,Friday
8,O0009,2026-03-13,2026,3,March,13,Friday
9,O0010,2026-03-05,2026,3,March,5,Thursday


## 7. Organizing the final processed DataFrame

The columns are arranged in a meaningful order and the rows are sorted by order date.

In [9]:
final_columns = [
    "Order_ID", "Order_Date", "Order_Year", "Order_Month", "Order_Month_Name",
    "Order_Day", "Day_of_Week", "Customer_ID", "Customer_Name",
    "Customer_Label", "City", "Region", "Membership_Type",
    "Product_ID", "Product_Name", "Category", "Brand", "Unit_Price",
    "Quantity", "Total_Sales", "Sales_Level", "Payment_Method", "Order_Status"
]

final_df = processed[final_columns].sort_values(
    by=["Order_Date", "Order_ID"]
).reset_index(drop=True)

display(final_df.head(10))
print("\nFinal shape:", final_df.shape)
print("\nMissing values:")
display(final_df.isna().sum().to_frame("Missing_Count"))


,Order_ID,Order_Date,Order_Year,Order_Month,Order_Month_Name,Order_Day,Day_of_Week,Customer_ID,Customer_Name,Customer_Label,City,Region,Membership_Type,Product_ID,Product_Name,Category,Brand,Unit_Price,Quantity,Total_Sales,Sales_Level,Payment_Method,Order_Status
0,O0051,2026-01-01,2026,1,January,1,Thursday,C008,Sara Ahmed,Sara Ahmed (Premium),Hyderabad,South,Premium,P017,Yoga Mat,Sports,FitLife,899,4,3596,High,Net Banking,Delivered
1,O0091,2026-01-01,2026,1,January,1,Thursday,C002,Zoya Khan,Zoya Khan (Regular),Delhi,North,Regular,P011,Air Fryer,Home & Kitchen,CookSmart,4999,2,9998,High,Credit Card,Delivered
2,O0022,2026-01-02,2026,1,January,2,Friday,C023,Nikhil Sood,Nikhil Sood (Premium),Chandigarh,North,Premium,P014,Data Science Handbook,Books,DataPress,899,4,3596,High,Net Banking,Delivered
3,O0106,2026-01-02,2026,1,January,2,Friday,C001,Aarav Sharma,Aarav Sharma (Premium),Srinagar,North,Premium,P009,Coffee Maker,Home & Kitchen,HomeBrew,3499,1,3499,High,Net Banking,Delivered
4,O0076,2026-01-03,2026,1,January,3,Saturday,C019,Yusuf Dar,Yusuf Dar (New),Srinagar,North,New,P015,Machine Learning Basics,Books,AIPress,999,1,999,Low,Debit Card,Delivered
5,O0041,2026-01-04,2026,1,January,4,Sunday,C008,Sara Ahmed,Sara Ahmed (Premium),Hyderabad,South,Premium,P012,Water Bottle,Home & Kitchen,HydroLife,699,5,3495,High,Cash on Delivery,Delivered
6,O0108,2026-01-04,2026,1,January,4,Sunday,C006,Ishita Gupta,Ishita Gupta (Premium),Bengaluru,South,Premium,P020,Dumbbell Set,Sports,StrongFit,1999,1,1999,Medium,Net Banking,Delivered
7,O0018,2026-01-05,2026,1,January,5,Monday,C003,Rohan Mehta,Rohan Mehta (Premium),Mumbai,West,Premium,P011,Air Fryer,Home & Kitchen,CookSmart,4999,4,19996,High,Net Banking,Cancelled
8,O0104,2026-01-05,2026,1,January,5,Monday,C011,Vivaan Kapoor,Vivaan Kapoor (New),Jaipur,North,New,P006,Hoodie,Clothing,UrbanWear,1599,3,4797,High,Debit Card,Shipped
9,O0110,2026-01-05,2026,1,January,5,Monday,C007,Aditya Verma,Aditya Verma (Regular),Pune,West,Regular,P005,Power Bank,Electronics,VoltPlus,1199,1,1199,Low,UPI,Delivered



Final shape: (120, 23)

Missing values:


,Missing_Count
Order_ID,0
Order_Date,0
Order_Year,0
Order_Month,0
Order_Month_Name,0
Order_Day,0
Day_of_Week,0
Customer_ID,0
Customer_Name,0
Customer_Label,0


## 8. Export the processed dataset

The final DataFrame is exported as `Day9_Processed_Ecommerce_Dataset.csv`.

In [10]:
output_file = DATA_DIR / "Day9_Processed_Ecommerce_Dataset.csv"

final_df.to_csv(output_file, index=False)

print(f"Processed dataset saved to: {output_file}")
print(f"Rows exported: {len(final_df)}")
print(f"Columns exported: {len(final_df.columns)}")


Processed dataset saved to: Day9_Processed_Ecommerce_Dataset.csv
Rows exported: 120
Columns exported: 23


## Conclusion

The three source datasets were successfully combined into one clean e-commerce dataset. The final dataset contains order, customer, and product information, calculated sales values, sales categories, customer labels, and DateTime-based features. It is ready for further analysis or visualization in Pandas, Matplotlib, or other data-analysis tools.